In [5]:
"""
=============================================================
 GNSS Position Error Correction  –  XGBoost Regression
 Equivalent of the MATLAB LSBoost script, using true XGBoost
=============================================================

 Input  : final_residuals_fused.mat  (or .csv — see LOAD section)
 Output : xgboost_gnss_model.joblib  +  results printed to console
          plots saved as PNG files

 Requirements:
    pip install xgboost scikit-learn scipy pandas numpy matplotlib joblib
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

from scipy.io        import loadmat
from scipy.stats     import gaussian_kde
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import mean_squared_error
from xgboost import XGBRegressor
import joblib

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
MAT_FILE    = '/mnt/c/UrbanNav/DeepUrban/mat/final_residuals_fused.mat'
OUTPUT_MODEL= '/mnt/c/UrbanNav/DeepUrban/mat/xgboost_gnss_model.joblib'
PLOT_DIR    = '/mnt/c/UrbanNav/DeepUrban/mat'   # PNGs saved here

# XGBoost hyper-parameters  (mirrors MATLAB: 200 trees, lr=0.05, depth~5)
XGB_PARAMS = dict(
    n_estimators      = 200,
    learning_rate     = 0.05,
    max_depth         = 5,          # ~MaxNumSplits=20 in a depth-5 tree
    subsample         = 0.8,        # row sampling per tree  (XGBoost extra)
    colsample_bytree  = 0.8,        # feature sampling per tree (XGBoost extra)
    min_child_weight  = 3,
    reg_lambda        = 1.0,        # L2 regularisation
    objective         = "reg:squarederror",
    eval_metric       = "rmse",
    tree_method       = "hist",     # fast histogram method
    random_state      = 42,
    n_jobs            = -1,         # use all CPU cores
)

K_FOLDS    = 5
TEST_SIZE  = 0.2
FEAT_NAMES = ["SNR", "Azimuth", "Elevation", "Doppler", "RangeResidual"]

# ──────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────────────────────
print("Loading data …")
mat   = loadmat(MAT_FILE, squeeze_me=True, struct_as_record=False)

# Extract the finalTable struct  ─  adjust key if your .mat uses a different name
ft = mat["finalTable"]

def field(name):
    """Pull a numeric field from the MATLAB table struct."""
    return np.array(ft.__dict__[name], dtype=float).flatten()

SNR          = field("SNR")
Azimuth      = field("Azimuth")
Elevation    = field("Elevation")
Doppler      = field("Doppler")
RangeResid   = field("RangeResidual")

dX = field("dX");   dY = field("dY");   dZ = field("dZ")
EstX = field("EstX"); EstY = field("EstY"); EstZ = field("EstZ")
GTX  = field("GT_X"); GTY  = field("GT_Y"); GTZ  = field("GT_Z")

# ──────────────────────────────────────────────────────────────
# 2. FEATURE MATRIX
# ──────────────────────────────────────────────────────────────
X_raw = np.column_stack([SNR, Azimuth, Elevation, Doppler, RangeResid])

# Remove rows with any NaN
valid = (
    np.all(np.isfinite(X_raw), axis=1) &
    np.isfinite(dX) & np.isfinite(dY) & np.isfinite(dZ)
)
X_raw = X_raw[valid]
dX, dY, dZ     = dX[valid], dY[valid], dZ[valid]
EstX, EstY, EstZ = EstX[valid], EstY[valid], EstZ[valid]
GTX,  GTY,  GTZ  = GTX[valid],  GTY[valid],  GTZ[valid]

print(f"Valid samples: {X_raw.shape[0]:,}")

# ──────────────────────────────────────────────────────────────
# 3. PART 1 — HOLD-OUT BASELINE  (80 / 20)
# ──────────────────────────────────────────────────────────────
print("\n── Part 1: Hold-out baseline ─────────────────────────")

idx_all = np.arange(len(dX))
tr, te  = train_test_split(idx_all, test_size=TEST_SIZE, random_state=42)

def make_pipeline():
    """
    Pipeline:  StandardScaler  →  XGBRegressor
    Scaler is fit ONLY on training fold  (fixes the leakage in the MATLAB script).
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("xgb",    XGBRegressor(**XGB_PARAMS)),
    ])

print("Training baseline models (X / Y / Z) …")
pipeX = make_pipeline().fit(X_raw[tr], dX[tr])
pipeY = make_pipeline().fit(X_raw[tr], dY[tr])
pipeZ = make_pipeline().fit(X_raw[tr], dZ[tr])

pred_dx_te = pipeX.predict(X_raw[te])
pred_dy_te = pipeY.predict(X_raw[te])
pred_dz_te = pipeZ.predict(X_raw[te])

rmse = lambda y, yh: np.sqrt(mean_squared_error(y, yh))
print(f"\nTEST RMSE:")
print(f"  dX: {rmse(dX[te], pred_dx_te):.3f} m")
print(f"  dY: {rmse(dY[te], pred_dy_te):.3f} m")
print(f"  dZ: {rmse(dZ[te], pred_dz_te):.3f} m")

err_before_te = np.sqrt((EstX[te]-GTX[te])**2 +
                        (EstY[te]-GTY[te])**2 +
                        (EstZ[te]-GTZ[te])**2)

err_after_te  = np.sqrt((EstX[te]-pred_dx_te-GTX[te])**2 +
                        (EstY[te]-pred_dy_te-GTY[te])**2 +
                        (EstZ[te]-pred_dz_te-GTZ[te])**2)

print(f"\nTEST 3D ERROR:")
print(f"  Mean BEFORE : {err_before_te.mean():.3f} m")
print(f"  Mean AFTER  : {err_after_te.mean():.3f} m")

# ──────────────────────────────────────────────────────────────
# 4. PART 2 — K-FOLD CROSS-VALIDATION  (no leakage)
# ──────────────────────────────────────────────────────────────
print(f"\n── Part 2: {K_FOLDS}-Fold Cross-Validation ───────────────────")

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

pred_dx_cv = np.zeros_like(dX)
pred_dy_cv = np.zeros_like(dY)
pred_dz_cv = np.zeros_like(dZ)

for fold, (tr_idx, te_idx) in enumerate(kf.split(X_raw), 1):
    print(f"  Fold {fold}/{K_FOLDS} …", end=" ")

    pX = make_pipeline().fit(X_raw[tr_idx], dX[tr_idx])
    pY = make_pipeline().fit(X_raw[tr_idx], dY[tr_idx])
    pZ = make_pipeline().fit(X_raw[tr_idx], dZ[tr_idx])

    pred_dx_cv[te_idx] = pX.predict(X_raw[te_idx])
    pred_dy_cv[te_idx] = pY.predict(X_raw[te_idx])
    pred_dz_cv[te_idx] = pZ.predict(X_raw[te_idx])

    # Per-fold 3D error (quick check)
    e_b = np.sqrt((EstX[te_idx]-GTX[te_idx])**2 +
                  (EstY[te_idx]-GTY[te_idx])**2 +
                  (EstZ[te_idx]-GTZ[te_idx])**2)
    e_a = np.sqrt((EstX[te_idx]-pred_dx_cv[te_idx]-GTX[te_idx])**2 +
                  (EstY[te_idx]-pred_dy_cv[te_idx]-GTY[te_idx])**2 +
                  (EstZ[te_idx]-pred_dz_cv[te_idx]-GTZ[te_idx])**2)
    print(f"3D before={e_b.mean():.2f} m  after={e_a.mean():.2f} m")

# ──────────────────────────────────────────────────────────────
# 5. FULL-DATASET RESULTS
# ──────────────────────────────────────────────────────────────
err_before_all = np.sqrt((EstX-GTX)**2 + (EstY-GTY)**2 + (EstZ-GTZ)**2)
err_after_all  = np.sqrt((EstX-pred_dx_cv-GTX)**2 +
                         (EstY-pred_dy_cv-GTY)**2 +
                         (EstZ-pred_dz_cv-GTZ)**2)

print("\n══ FULL DATASET RESULTS (cross-validated) ══════════════")
print(f"  Mean  BEFORE : {err_before_all.mean():.3f} m")
print(f"  Mean  AFTER  : {err_after_all.mean():.3f} m")
print(f"  RMSE  BEFORE : {np.sqrt(np.mean(err_before_all**2)):.3f} m")
print(f"  RMSE  AFTER  : {np.sqrt(np.mean(err_after_all**2)):.3f} m")
print(f"  95th% BEFORE : {np.percentile(err_before_all, 95):.3f} m")
print(f"  95th% AFTER  : {np.percentile(err_after_all,  95):.3f} m")
print(f"  Improvement  : {(1 - err_after_all.mean()/err_before_all.mean())*100:.1f}%")

# ──────────────────────────────────────────────────────────────
# 6. TRAIN FINAL MODELS ON FULL DATA  (for deployment)
# ──────────────────────────────────────────────────────────────
print("\nTraining final models on full dataset for export …")
final_pipeX = make_pipeline().fit(X_raw, dX)
final_pipeY = make_pipeline().fit(X_raw, dY)
final_pipeZ = make_pipeline().fit(X_raw, dZ)

joblib.dump(
    {"mdlX": final_pipeX, "mdlY": final_pipeY, "mdlZ": final_pipeZ},
    OUTPUT_MODEL
)
print(f"Models saved → {OUTPUT_MODEL}")

# ──────────────────────────────────────────────────────────────
# 7. FEATURE IMPORTANCE  (from final X model)
# ──────────────────────────────────────────────────────────────
importance = final_pipeX.named_steps["xgb"].feature_importances_

# ──────────────────────────────────────────────────────────────
# 8. PLOTS
# ──────────────────────────────────────────────────────────────
import os
plt.rcParams.update({
    "figure.facecolor": "#0d0d14",
    "axes.facecolor":   "#0d0d14",
    "axes.edgecolor":   "#2a2a3a",
    "axes.labelcolor":  "#c8c8d8",
    "xtick.color":      "#888899",
    "ytick.color":      "#888899",
    "text.color":       "#c8c8d8",
    "grid.color":       "#1e1e2e",
    "grid.linewidth":   0.6,
    "font.family":      "monospace",
    "font.size":        10,
})

COL_BEFORE = "#e05c5c"
COL_AFTER  = "#4fc97e"
N = len(err_before_all)

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor("#0d0d14")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# ── Plot 1: Time-series ───────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(err_before_all, color=COL_BEFORE, lw=0.7, alpha=0.85, label="Before correction")
ax1.plot(err_after_all,  color=COL_AFTER,  lw=0.7, alpha=0.85, label="After correction")
ax1.axhline(err_before_all.mean(), color=COL_BEFORE, lw=1.2, ls="--", alpha=0.5)
ax1.axhline(err_after_all.mean(),  color=COL_AFTER,  lw=1.2, ls="--", alpha=0.5)
ax1.set_title("3D Position Error  –  Time Series (full cross-validated dataset)", fontsize=11)
ax1.set_xlabel("Sample index"); ax1.set_ylabel("Error (m)")
ax1.legend(framealpha=0.15, loc="upper right")
ax1.grid(True)

# ── Plot 2: Feature importance ────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
colors = plt.cm.plasma(np.linspace(0.3, 0.9, len(FEAT_NAMES)))
bars = ax2.barh(FEAT_NAMES, importance[np.argsort(importance)],
                color=colors[np.argsort(importance)], edgecolor="none")
ax2.set_title("XGBoost Feature Importance\n(dX model, F-score)", fontsize=11)
ax2.set_xlabel("Importance")
ax2.invert_yaxis()
for bar, val in zip(bars, importance[np.argsort(importance)]):
    ax2.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=8, color="#aaaacc")
ax2.grid(True, axis="x")

# ── Plot 3: Box plot ──────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
bp = ax3.boxplot(
    [err_before_all, err_after_all],
    labels=["Before", "After"],
    patch_artist=True,
    medianprops=dict(color="white", lw=2),
    whiskerprops=dict(color="#888899"),
    capprops=dict(color="#888899"),
    flierprops=dict(marker=".", color="#555566", markersize=2),
)
bp["boxes"][0].set_facecolor(COL_BEFORE + "55")
bp["boxes"][1].set_facecolor(COL_AFTER  + "55")
ax3.set_title("Error Distribution", fontsize=11)
ax3.set_ylabel("3D Error (m)"); ax3.grid(True, axis="y")

# ── Plot 4: CDF ───────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for err, col, lbl in [(err_before_all, COL_BEFORE, "Before"),
                       (err_after_all,  COL_AFTER,  "After")]:
    sorted_e = np.sort(err)
    cdf = np.arange(1, len(sorted_e)+1) / len(sorted_e)
    ax4.plot(sorted_e, cdf, color=col, lw=1.8, label=lbl)
    # 50th and 95th percentile markers
    for pct, ls in [(50,"--"),(95,":")]:
        v = np.percentile(err, pct)
        ax4.axvline(v, color=col, lw=0.8, ls=ls, alpha=0.6)
ax4.set_title("CDF of 3D Position Error", fontsize=11)
ax4.set_xlabel("Error (m)"); ax4.set_ylabel("Cumulative probability")
ax4.legend(framealpha=0.15); ax4.grid(True)
ax4.set_ylim(0, 1)

# ── Plot 5: Scatter true vs predicted residuals (dX) ─────────
ax5 = fig.add_subplot(gs[1, 2])
lim = max(abs(dX).max(), abs(pred_dx_cv).max()) * 1.05
ax5.scatter(dX, pred_dx_cv, s=2, alpha=0.3,
            c=np.abs(dX - pred_dx_cv), cmap="plasma", rasterized=True)
ax5.plot([-lim, lim], [-lim, lim], color="#aaaacc", lw=1, ls="--", alpha=0.6)
ax5.set_xlim(-lim, lim); ax5.set_ylim(-lim, lim)
ax5.set_title("True vs Predicted  dX  (CV)", fontsize=11)
ax5.set_xlabel("True dX (m)"); ax5.set_ylabel("Predicted dX (m)")
ax5.grid(True)

# ── Suptitle ──────────────────────────────────────────────────
fig.suptitle(
    f"GNSS XGBoost Correction  |  n={N:,} samples  |  "
    f"Mean error  {err_before_all.mean():.2f} m → {err_after_all.mean():.2f} m  "
    f"({(1-err_after_all.mean()/err_before_all.mean())*100:.1f}% improvement)",
    fontsize=13, y=0.98, color="#e8e8f8"
)

plot_path = os.path.join(PLOT_DIR, "gnss_xgboost_results.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"\nPlot saved → {plot_path}")
print("\nDone ✔")

Loading data …


KeyError: 'finalTable'